# Workshop 2 · Power BI Dataset Performance · Solution

![Power BI report mockup](../../../assets/images/powerbi_report_mockup.png)

This is the reference implementation for the Day 2 serving layer: a monthly aggregate, an incremental-refresh view and a KPI table for Power BI.


## Business Scenario

RetailHub needs fast executive BI and safe drill-through. The solution keeps the Day 1 Gold model intact and adds serving objects optimized for Power BI consumption.


## Target Objects

| Object | Grain | Purpose |
|---|---|---|
| `gold.fact_sales_dashboard_monthly` | month x region x category x channel | Import-friendly summary pages |
| `gold.v_fact_sales_incremental` | one row per line | drill-through and incremental refresh |
| `gold.kpi_monthly` | one row per month | KPI cards and refresh sanity checks |


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.


In [ ]:
%run ../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before starting the workshop.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

Day 2 starts from the Gold model created in Workshop 1. This check fails early if the model is not available.


In [ ]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run Workshop 1 solution before starting Day 2.")
print("[OK] Day 1 Gold model is available.")


### Workshop Validation Helpers

These helper functions keep the task-level assertions short and readable. They are used only for validation, not for the main SQL implementation.


In [ ]:
# -- Validation helpers --
def _table_exists(full_name):
    return spark.catalog.tableExists(full_name)


def _require_table(full_name):
    assert _table_exists(full_name), f"Missing object: {full_name}"


def _columns(full_name):
    return set(spark.table(full_name).columns)


def _require_columns(full_name, required_columns):
    missing = sorted(set(required_columns) - _columns(full_name))
    assert not missing, f"{full_name} missing columns: {missing}"


def _first(query):
    row = spark.sql(query).first()
    assert row is not None, "Query returned no rows"
    return row


def _print_ok(message):
    print(message)


## 1. Source Baseline


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM gold.fact_sales_dashboard


## 2. Build `gold.fact_sales_dashboard_monthly`

The aggregate is intentionally smaller than the line-grain table and keeps the business grain required by executive visuals.


Solution note: this table intentionally trades detail for speed. It is suitable for Import summary pages because the grain matches common executive visuals.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.fact_sales_dashboard_monthly
COMMENT 'Power BI monthly serving table: completed sales at year_month x customer_region x category x channel grain.'
AS
SELECT
  year_month,
  to_date(concat(year_month, '-01')) AS order_month,
  customer_region,
  category,
  channel,
  COUNT(*) AS line_rows,
  COUNT(DISTINCT order_id) AS completed_orders,
  SUM(quantity) AS quantity,
  ROUND(SUM(line_revenue), 2) AS revenue,
  ROUND(SUM(line_margin), 2) AS gross_margin,
  ROUND(100.0 * SUM(line_margin) / NULLIF(SUM(line_revenue), 0), 2) AS margin_rate_pct
FROM gold.fact_sales_dashboard
WHERE is_completed
GROUP BY year_month, customer_region, category, channel


Why this works: the monthly aggregate reduces scan volume for summary pages while keeping a clear, documented grain.


In [ ]:
%sql
SELECT
  COUNT(*) AS aggregate_rows,
  COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS distinct_grain_keys,
  COUNT(*) - COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS duplicate_grain_keys,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.fact_sales_dashboard_monthly


In [ ]:
# -- Validation --
table = f"{GOLD}.fact_sales_dashboard_monthly"
_require_table(table)
_require_columns(table, ["year_month", "order_month", "customer_region", "category", "channel", "line_rows", "completed_orders", "quantity", "revenue", "gross_margin", "margin_rate_pct"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS duplicate_grain_keys, SUM(CASE WHEN line_rows <= 0 THEN 1 ELSE 0 END) AS invalid_line_rows
FROM {GOLD}.fact_sales_dashboard_monthly
""")
assert row["rows"] > 0, "Task 1 failed: monthly aggregate is empty"
assert row["duplicate_grain_keys"] == 0, f"Task 1 failed: duplicate monthly grain keys = {row['duplicate_grain_keys']}"
assert row["invalid_line_rows"] == 0, f"Task 1 failed: invalid line_rows buckets = {row['invalid_line_rows']}"
_print_ok(f"Task 1 OK: monthly aggregate has {row['rows']} unique grain rows.")


## 3. Reconcile Aggregate to Detail


Solution note: reconciliation protects trust. A fast aggregate is not usable in Power BI unless it agrees with the certified detail source for the same KPI rule.

In [ ]:
%sql
WITH detail AS (
  SELECT
    ROUND(SUM(line_revenue), 2) AS detail_revenue,
    ROUND(SUM(line_margin), 2) AS detail_margin,
    COUNT(DISTINCT order_id) AS detail_completed_orders
  FROM gold.fact_sales_dashboard
  WHERE is_completed
),
aggregate AS (
  SELECT
    ROUND(SUM(revenue), 2) AS aggregate_revenue,
    ROUND(SUM(gross_margin), 2) AS aggregate_margin,
    SUM(completed_orders) AS aggregate_completed_order_bucket_count
  FROM gold.fact_sales_dashboard_monthly
)
SELECT
  detail_revenue,
  aggregate_revenue,
  ROUND(detail_revenue - aggregate_revenue, 2) AS revenue_diff,
  detail_margin,
  aggregate_margin,
  ROUND(detail_margin - aggregate_margin, 2) AS margin_diff,
  detail_completed_orders,
  aggregate_completed_order_bucket_count
FROM detail
CROSS JOIN aggregate


In [ ]:
# -- Validation --
row = _first(f"""
WITH detail AS (SELECT ROUND(SUM(line_revenue), 2) AS revenue FROM {GOLD}.fact_sales_dashboard WHERE is_completed), aggregate AS (SELECT ROUND(SUM(revenue), 2) AS revenue FROM {GOLD}.fact_sales_dashboard_monthly)
SELECT ROUND(detail.revenue - aggregate.revenue, 2) AS revenue_diff FROM detail CROSS JOIN aggregate
""")
assert row["revenue_diff"] is not None, "Task 2 failed: revenue diff is null"
assert abs(float(row["revenue_diff"])) < 0.01, f"Task 2 failed: aggregate/detail revenue diff = {row['revenue_diff']}"
_print_ok("Task 2 OK: monthly aggregate reconciles to completed detail revenue.")


Note: `aggregate_completed_order_bucket_count` can be higher than the global distinct order count because an order can contribute to multiple category/region/channel buckets. Revenue and margin must reconcile exactly; distinct orders are not additive across arbitrary buckets.


## 4. Build `gold.v_fact_sales_incremental`


Solution note: the view keeps drill-through detail and adds `order_datetime` so Power BI can apply `RangeStart` and `RangeEnd` filters cleanly.

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.v_fact_sales_incremental
COMMENT 'Power BI incremental-refresh view over line-grain dashboard table with order_datetime timestamp.'
AS
SELECT
  *,
  CAST(order_date AS TIMESTAMP) AS order_datetime
FROM gold.fact_sales_dashboard


Why this works: the view keeps line-grain detail and adds the timestamp column required by Power BI incremental refresh filters.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  MIN(order_datetime) AS min_order_datetime,
  MAX(order_datetime) AS max_order_datetime
FROM gold.v_fact_sales_incremental


In [ ]:
# -- Validation --
table = f"{GOLD}.v_fact_sales_incremental"
_require_table(table)
_require_columns(table, ["line_id", "order_date", "order_datetime", "line_revenue", "is_completed"])
row = _first(f"""
SELECT (SELECT COUNT(*) FROM {GOLD}.fact_sales_dashboard) AS dashboard_rows, COUNT(*) AS incremental_rows, SUM(CASE WHEN order_datetime IS NULL THEN 1 ELSE 0 END) AS null_order_datetime
FROM {GOLD}.v_fact_sales_incremental
""")
assert row["incremental_rows"] == row["dashboard_rows"], f"Task 3 failed: incremental rows {row['incremental_rows']} do not match dashboard rows {row['dashboard_rows']}"
assert row["null_order_datetime"] == 0, f"Task 3 failed: null order_datetime rows = {row['null_order_datetime']}"
_print_ok(f"Task 3 OK: incremental view exposes order_datetime for {row['incremental_rows']} detail rows.")


## 5. Test Incremental Refresh Window


Solution note: the half-open interval prevents overlapping refresh partitions, which is essential for stable incremental refresh.

In [ ]:
%sql
WITH params AS (
  SELECT
    TIMESTAMP '2024-01-01 00:00:00' AS RangeStart,
    TIMESTAMP '2024-02-01 00:00:00' AS RangeEnd
)
SELECT
  p.RangeStart,
  p.RangeEnd,
  COUNT(*) AS rows_in_window,
  MIN(v.order_date) AS min_order_date,
  MAX(v.order_date) AS max_order_date,
  ROUND(SUM(CASE WHEN v.is_completed THEN v.line_revenue ELSE 0 END), 2) AS completed_revenue
FROM gold.v_fact_sales_incremental v
CROSS JOIN params p
WHERE v.order_datetime >= p.RangeStart
  AND v.order_datetime < p.RangeEnd
GROUP BY p.RangeStart, p.RangeEnd


In [ ]:
# -- Validation --
row = _first(f"""
WITH params AS (SELECT TIMESTAMP '2024-01-01 00:00:00' AS RangeStart, TIMESTAMP '2024-02-01 00:00:00' AS RangeEnd), windowed AS (SELECT v.* FROM {GOLD}.v_fact_sales_incremental v CROSS JOIN params p WHERE v.order_datetime >= p.RangeStart AND v.order_datetime < p.RangeEnd)
SELECT COUNT(*) AS rows_in_window, MIN(order_datetime) AS min_order_datetime, MAX(order_datetime) AS max_order_datetime, SUM(CASE WHEN order_datetime >= TIMESTAMP '2024-02-01 00:00:00' THEN 1 ELSE 0 END) AS rows_on_or_after_range_end
FROM windowed
""")
assert row["rows_in_window"] > 0, "Task 4 failed: test refresh window returned no rows"
assert row["rows_on_or_after_range_end"] == 0, "Task 4 failed: half-open interval includes RangeEnd boundary rows"
_print_ok(f"Task 4 OK: half-open incremental window returned {row['rows_in_window']} rows and excludes RangeEnd.")


## 6. Build `gold.kpi_monthly`


Solution note: monthly KPI cards should read a compact table instead of scanning line-grain detail for every report interaction.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.kpi_monthly
COMMENT 'Monthly KPI table for Power BI cards and refresh sanity checks.'
AS
SELECT
  year_month,
  to_date(concat(year_month, '-01')) AS order_month,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  COUNT(DISTINCT CASE WHEN is_returned THEN order_id END) AS returned_orders,
  ROUND(
    100.0 * SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END)
    / NULLIF(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 0),
    2
  ) AS margin_rate_pct,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN is_returned THEN order_id END)
    / NULLIF(
        COUNT(DISTINCT CASE WHEN is_completed THEN order_id END)
        + COUNT(DISTINCT CASE WHEN is_returned THEN order_id END),
        0
      ),
    2
  ) AS return_rate_pct,
  ROUND(
    SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END)
    / NULLIF(COUNT(DISTINCT CASE WHEN is_completed THEN order_id END), 0),
    2
  ) AS avg_order_value
FROM gold.fact_sales_dashboard
GROUP BY year_month


Why this works: KPI cards can read a compact monthly table instead of repeatedly scanning detail rows.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT year_month) AS distinct_months,
  COUNT(*) - COUNT(DISTINCT year_month) AS duplicate_months,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.kpi_monthly


In [ ]:
# -- Validation --
table = f"{GOLD}.kpi_monthly"
_require_table(table)
_require_columns(table, ["year_month", "order_month", "revenue", "gross_margin", "completed_orders", "returned_orders", "margin_rate_pct", "return_rate_pct", "avg_order_value"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT year_month) AS duplicate_months, SUM(CASE WHEN revenue IS NULL OR gross_margin IS NULL OR completed_orders IS NULL THEN 1 ELSE 0 END) AS null_core_kpis, SUM(CASE WHEN margin_rate_pct IS NOT NULL AND (margin_rate_pct < -100 OR margin_rate_pct > 100) THEN 1 ELSE 0 END) AS suspicious_margin_rates
FROM {GOLD}.kpi_monthly
""")
assert row["rows"] > 0, "Task 5 failed: kpi_monthly is empty"
assert row["duplicate_months"] == 0, f"Task 5 failed: duplicate year_month rows = {row['duplicate_months']}"
assert row["null_core_kpis"] == 0, f"Task 5 failed: rows with null core KPIs = {row['null_core_kpis']}"
assert row["suspicious_margin_rates"] == 0, f"Task 5 failed: suspicious margin rate rows = {row['suspicious_margin_rates']}"
_print_ok(f"Task 5 OK: kpi_monthly has {row['rows']} monthly KPI rows with valid grain.")


## 7. KPI Smoke Test


In [ ]:
%sql
SELECT *
FROM gold.kpi_monthly
ORDER BY year_month DESC
LIMIT 12


## 8. BI Contract


Solution note: this contract tells a Power BI developer which object to use for summary pages, drill-through and KPI cards before opening Power BI Desktop.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW w2_bi_contract AS
SELECT
  'gold.fact_sales_dashboard_monthly' AS object_name,
  'year_month x customer_region x category x channel' AS grain,
  'Import' AS recommended_mode,
  'refresh after Gold refresh completes' AS refresh_expectation,
  'executive summary pages and trend visuals' AS primary_use
UNION ALL
SELECT
  'gold.v_fact_sales_incremental',
  'line_id',
  'Import with incremental refresh or controlled DirectQuery',
  'date-window refresh using order_datetime',
  'drill-through and detail validation'
UNION ALL
SELECT
  'gold.kpi_monthly',
  'year_month',
  'Import',
  'refresh after monthly aggregate',
  'KPI cards and refresh sanity checks'


Why this works: the BI contract turns source-object choice into an explicit handoff artifact for report developers.


In [ ]:
%sql
SELECT *
FROM w2_bi_contract
ORDER BY object_name


In [ ]:
# -- Validation --
row = _first("""
SELECT COUNT(*) AS rows, COUNT(DISTINCT object_name) AS distinct_objects, SUM(CASE WHEN grain IS NULL OR recommended_mode IS NULL OR refresh_expectation IS NULL OR primary_use IS NULL THEN 1 ELSE 0 END) AS incomplete_rows
FROM w2_bi_contract
""")
expected_objects = {"gold.fact_sales_dashboard_monthly", "gold.v_fact_sales_incremental", "gold.kpi_monthly"}
observed_objects = {r["object_name"] for r in spark.sql("SELECT object_name FROM w2_bi_contract").collect()}
missing = sorted(expected_objects - observed_objects)
assert row["rows"] >= 3, "Task 6 failed: BI contract must contain at least three serving objects"
assert not missing, f"Task 6 failed: BI contract missing objects: {missing}"
assert row["incomplete_rows"] == 0, f"Task 6 failed: incomplete BI contract rows = {row['incomplete_rows']}"
_print_ok("Task 6 OK: BI contract covers the W2 serving objects with grain, mode, refresh expectation and use case.")


## 9. Quality Gate

This assertion cell is intentionally Python because it turns the workshop checks into a hard stop.


In [ ]:
checks = []
checks_spec = {
    "monthly_duplicate_grain_keys": {
        "tolerance": 0,
        "query": """
            SELECT COUNT(*) - COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS n
            FROM gold.fact_sales_dashboard_monthly
        """,
    },
    "kpi_duplicate_months": {
        "tolerance": 0,
        "query": """
            SELECT COUNT(*) - COUNT(DISTINCT year_month) AS n
            FROM gold.kpi_monthly
        """,
    },
    "incremental_duplicate_lines": {
        "tolerance": 0,
        "query": """
            SELECT COUNT(*) - COUNT(DISTINCT line_id) AS n
            FROM gold.v_fact_sales_incremental
        """,
    },
    "monthly_revenue_diff_abs": {
        "tolerance": 1.00,
        "query": """
            WITH d AS (
              SELECT ROUND(SUM(line_revenue), 2) AS revenue
              FROM gold.fact_sales_dashboard
              WHERE is_completed
            ),
            a AS (
              SELECT ROUND(SUM(revenue), 2) AS revenue
              FROM gold.fact_sales_dashboard_monthly
            )
            SELECT ABS(ROUND(d.revenue - a.revenue, 2)) AS n
            FROM d CROSS JOIN a
        """,
    },
}

for name, spec in checks_spec.items():
    observed = float(spark.sql(spec["query"]).first()["n"] or 0)
    tolerance = float(spec["tolerance"])
    status = "OK" if observed <= tolerance else "FAIL"
    checks.append((name, observed, tolerance, status))

failed = [name for name, observed, tolerance, status in checks if status == "FAIL"]
if failed:
    raise Exception(f"Workshop 2 quality gate failed: {failed}")

display(spark.createDataFrame(checks, ["check_name", "observed_value", "tolerance", "status"]))


## Completion

Workshop 2 serving objects are now ready for Day2_02 performance analysis and Day2_03 AI/BI datasets.
